# Prefill vs Decode Latency Breakdown

This notebook measures the time spent in each phase for varying prompt lengths
and output lengths, using the local GPT-2 backend.

**What to expect:**
- Prefill time scales with **prompt length** (processes all tokens in parallel).
- Decode time scales with **output length** (one forward pass per token).
- Disaggregation is most valuable when prefill >> decode per step (long prompts,
  short outputs) *or* when you have many concurrent short-prompt requests.


In [ ]:
import time
import sys
sys.path.insert(0, "..")

import torch
import matplotlib.pyplot as plt
import numpy as np

from workers.prefill.worker import PrefillWorker
from workers.decode.worker import DecodeWorker
from kv_transfer.serializer import deserialize_kv_cache

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Load both workers once
prefill_worker = PrefillWorker(device=DEVICE)
prefill_worker.load()

decode_worker = DecodeWorker(device=DEVICE)
decode_worker.load()

In [ ]:
def measure_prefill(prompt: str, n_runs: int = 5) -> float:
    """Return mean prefill latency in milliseconds over n_runs."""
    # TODO: warm up the first run before timing
    latencies = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        kv_b64, input_ids, _ = prefill_worker.prefill(prompt)
        latencies.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(latencies)), kv_b64, input_ids


def measure_decode(kv_b64: str, input_ids: list, max_new_tokens: int, n_runs: int = 3) -> float:
    """Return mean decode latency in milliseconds over n_runs."""
    # TODO: account for token-level latency (latency / generated_tokens)
    latencies = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        decode_worker.decode(kv_b64, input_ids, max_new_tokens=max_new_tokens)
        latencies.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(latencies))

In [ ]:
# ── Sweep over prompt lengths ─────────────────────────────────────────────────
# TODO: replace with real prompts from a dataset (e.g. ShareGPT)

base_word = "hello "
prompt_lengths = [16, 32, 64, 128, 256]  # approximate token counts
decode_tokens = 32

results = []
for n in prompt_lengths:
    prompt = base_word * n
    p_lat, kv_b64, input_ids = measure_prefill(prompt)
    d_lat = measure_decode(kv_b64, input_ids, max_new_tokens=decode_tokens)
    results.append({"prompt_tokens": n, "prefill_ms": p_lat, "decode_ms": d_lat})
    print(f"prompt~{n} tokens | prefill {p_lat:.1f} ms | decode {d_lat:.1f} ms")

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
xs = [r["prompt_tokens"] for r in results]
prefill_ms = [r["prefill_ms"] for r in results]
decode_ms  = [r["decode_ms"]  for r in results]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs, prefill_ms, marker="o", label=f"prefill")
ax.plot(xs, decode_ms,  marker="s", label=f"decode ({decode_tokens} tokens)")
ax.set_xlabel("Approximate prompt tokens")
ax.set_ylabel("Latency (ms)")
ax.set_title("Prefill vs Decode latency — GPT-2, " + DEVICE)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("latency_breakdown.png", dpi=150)
plt.show()

## TODO: Next experiments

- [ ] Measure KV cache transfer overhead (serialize + HTTP round-trip).
- [ ] Compare token-level decode latency: disaggregated vs. co-located.
- [ ] Sweep output length while holding prompt fixed.
- [ ] Test with a larger model (GPT-2 medium / large) to see if the ratio changes.
- [ ] Add concurrency: 4 simultaneous requests to observe head-of-line blocking.